# Trabajo Práctico Aprendizaje Automático 1

In [7]:
import pandas as pd
import numpy as np


Fijamos una semilla para tener reproducibilidad en los resultados

In [8]:
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Separación de datos

Primero que nada guardamos los datos que vamos a usar en el trabajo.

In [9]:
data = pd.read_csv('../Data/data.csv')

Luego lo que debemos hacer antes de empezar con el análisis exploratorio y el desarrollo de modelos es, dividir nuestros datos en desarrollo y control. Esto se hace con el objetivo de que al final del trabajo podamos reportar la performance esperada del modelo final con datos de la vida real.

Para hacer esto se me ocurrieron dos ideas (nada innovador) -> mis 2 huevos vas a innovar (firma giannisluca a sebasouto):

1. Mezclar todos los datos y quedarnos con el porcentaje deseado.

2. Mezclar todos los datos y quedarnos con el porcentaje deseado pero estratificando. Nose si hacer esto es trampa, porque haciendo eso ya voy a conocer algo de los datos de control. 

Voy a implementar ambos y despues decidimos. IMPORTANTE: antes de seguir con los otros puntos elegir una forma de dividir y no volver a tocar esos datos.

In [10]:
#Defino el porcentaje que vamos a usar para control. A debatir.. me pareció mucho usar el 20%. 

porcent = 0.10

### Opción 1:

In [11]:
# Mezclamos los datos directamente del dataframe. 
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

#Tomamos el porcentaje de control y desarrollo
n_control = int(len(data)*porcent)

#Dividimos..
data_control = data.iloc[:n_control].reset_index(drop=True)
data_dev = data.iloc[n_control:].reset_index(drop=True)

### Opción 2:

In [12]:
#Primero obtenemos los índices de cada clase en el data frame, ya que queremos estratificar. Tuve que agregar los .copy() porque me tiraba error.
indices_pos = data[data['target'] == 1].index.to_numpy().copy()
indices_neg = data[data['target'] == 0].index.to_numpy().copy()

# Luego mezclamos los índices de cada clase. 
rng.shuffle(indices_pos)
rng.shuffle(indices_neg)

# Calculamos la cantidad de instancias de control. ACA es la parte donde se estratifica.
n_control_pos = int(np.round(len(indices_pos) * porcent))
n_control_neg = int(np.round(len(indices_neg) * porcent))

# Partimos los indices de cada clase en control y desarrollo.
idx_control_pos = indices_pos[:n_control_pos]
idx_dev_pos     = indices_pos[n_control_pos:]

idx_control_neg = indices_neg[:n_control_neg]
idx_dev_neg     = indices_neg[n_control_neg:]

#Concatenamos los indices de control y desarrollo de cada clase para obtener los índices finales.
idx_control = np.concatenate([idx_control_pos, idx_control_neg])
idx_dev     = np.concatenate([idx_dev_pos, idx_dev_neg])

# Y mezclamos para que no nos queden los positivos por un lado y los negativos por el otro. Esto nose si es al pedo
rng.shuffle(idx_control)
rng.shuffle(idx_dev)

# Finalmente construimos los dataframes de desarrollo y control 
data_dev = data.loc[idx_dev].reset_index(drop=True)
data_control = data.loc[idx_control].reset_index(drop=True)

# Verificación de la estratificación
print(f"Total de datos: {len(data)}")
print(f"Desarrollo: {len(data_dev)} filas | Proporción Positivos: {data_dev['target'].mean():.4f}")
print(f"Control:    {len(data_control)} filas  | Proporción Positivos: {data_control['target'].mean():.4f}")

Total de datos: 500
Desarrollo: 450 filas | Proporción Positivos: 0.2822
Control:    50 filas  | Proporción Positivos: 0.2800


## 2: Construcción de modelos

### 2.1: Entrenar arbol de desición

Primero, armamos los train con la data ya separada por Desarrollo y Control:

In [13]:
X_train = data_dev.drop(columns=['target'])
y_train = data_dev['target']

Entrenamos un arbol de desición con altura maxima 3 e hiperparam por defecto

In [14]:
from sklearn.tree import DecisionTreeClassifier

In [15]:
#este modelo por ahora no se usa para nada, es la base
mode_full_train = DecisionTreeClassifier(max_depth=3)
mode_full_train.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=3)

### 2.2: Evaluar K-Fold

Hago folds estratificados por desbalance

In [16]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, auc
)



In [17]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=912)#comoteduelelaco...

kf por dentro tiene indexaciones por cada fold. Es decir, se ve algo asi:

fold 1: ((0,1,2,5),(3,4)) -> entreno con train[0] , train[1] ,..., y evaluo con 3,4. (escribir esto lindo)

In [ ]:


fold = 1 #negrada, puede quedar mejor esto si se itera con un for pero queda bastante mas dificil de leer.
         #lo voy aumentando por cada for (es para printear boludeces nomas)
         #TODO: borrar la parte que dice -negrada- para el proximo que lea esto!

#resultados fold quedan en este result_folds en orden, no es muy lindo. TODO: si se les ocurre como ponerlo mas lindo cambienlo
results_folds = []

y_pred_global_folds = np.empty(len(y_train)) #para score global
y_prob_global_folds = np.empty(len(y_train)) #para score global

for train_idx, val_idx in kf.split(X_train, y_train): #train idx: indices para entrenar fold i, val idx: indices para testear fold i


    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx] #agarro features train y val para el fold i

    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx] #agarro labels train y val para el fold i

    #lo sig como chequeo, not nescesario
    
    print(f"Cantidad de datos en el fold {fold} de train: {len(y_train_fold)}")
    print(f"Cantidad de datos en el fold {fold} de validation: {len(y_val_fold)}")
    print(f"Proporcion de datos positivos en el fold {fold} de train: {y_train_fold.mean():.4f}")
    print(f"Proporcion de datos positivos en el fold {fold} de validation: {y_val_fold.mean():.4f}")


    model = DecisionTreeClassifier(max_depth=3)
    model.fit(X_train_fold, y_train_fold) #entreno modelo en datos fold


    y_pred_train_fold = model.predict(X_train_fold) #predicciones del modelo entrenado en los datos de train fold
    y_pred_val_fold = model.predict(X_val_fold) #predicciones del modelo entrenado en los datos de validación fold

    y_prob_train_fold = model.predict_proba(X_train_fold)[:, 1] #probabilidades de la clase positiva para los datos de train fold
    y_prob_val_fold = model.predict_proba(X_val_fold)[:, 1] #probabilidades de la clase positiva para los datos de validación fold

    #Guardo las predicciones y probabilidades de las validaciones en su posc original, para luego calcular global
    y_pred_global[val_idx] = y_pred_val_fold
    y_prob_global[val_idx] = y_prob_val_fold

    ###Metricas de performance para cada fold

    # AUPRC train
    precision_train, recall_train, _ = precision_recall_curve(
        y_train_fold,
        y_prob_train_fold
    )
    auprc_train = auc(recall_train, precision_train)

    # AUPRC validation
    precision_val, recall_val, _ = precision_recall_curve(
        y_val_fold,
        y_prob_val_fold
    )
    auprc_val = auc(recall_val, precision_val)


    results_folds.append({
        "Accuracy train": accuracy_score(y_train_fold, y_pred_train_fold),
        "Accuracy validation": accuracy_score(y_val_fold, y_pred_val_fold),
        "AUPRC train": auprc_train,
        "AUPRC validation": auprc_val,
        "AUCROC train": roc_auc_score(y_train_fold, y_prob_train_fold),
        "AUCROC validation": roc_auc_score(y_val_fold, y_prob_val_fold),
    })

    fold += 1


#le puse de nombres validación, no se si es ese o test, creo q en la practica lo habiamos llamado val
#same shi




Cantidad de datos en el fold 1 de train: 360
Cantidad de datos en el fold 1 de validation: 90
Proporcion de datos positivos en el fold 1 de train: 0.2806
Proporcion de datos positivos en el fold 1 de validation: 0.2889
Cantidad de datos en el fold 2 de train: 360
Cantidad de datos en el fold 2 de validation: 90
Proporcion de datos positivos en el fold 2 de train: 0.2806
Proporcion de datos positivos en el fold 2 de validation: 0.2889
Cantidad de datos en el fold 3 de train: 360
Cantidad de datos en el fold 3 de validation: 90
Proporcion de datos positivos en el fold 3 de train: 0.2833
Proporcion de datos positivos en el fold 3 de validation: 0.2778
Cantidad de datos en el fold 4 de train: 360
Cantidad de datos en el fold 4 de validation: 90
Proporcion de datos positivos en el fold 4 de train: 0.2833
Proporcion de datos positivos en el fold 4 de validation: 0.2778
Cantidad de datos en el fold 5 de train: 360
Cantidad de datos en el fold 5 de validation: 90
Proporcion de datos positivos 

In [23]:
results_df = pd.DataFrame(results_folds)

#esto de aca abajo no lo tengo clarisimo tengo sueño mañana lo veo bien, es solo para ver resultados ahora
results_df.index = range(1, len(results_df) + 1)
results_df.index.name = "Fold"

display(results_df.round(3))

,Accuracy train,Accuracy validation,AUPRC train,AUPRC validation,AUCROC train,AUCROC validation
Fold,,,,,,
1,0.803,0.700,0.705,0.369,0.803,0.579
2,0.831,0.700,0.755,0.352,0.733,0.579
3,0.803,0.778,0.759,0.428,0.841,0.700
4,0.833,0.778,0.764,0.586,0.820,0.716
5,0.850,0.722,0.779,0.399,0.789,0.597


In [24]:
#agrego promedios
results_df.loc["Promedio"] = results_df.mean()
display(results_df.round(3))

,Accuracy train,Accuracy validation,AUPRC train,AUPRC validation,AUCROC train,AUCROC validation
Fold,,,,,,
1,0.803,0.700,0.705,0.369,0.803,0.579
2,0.831,0.700,0.755,0.352,0.733,0.579
3,0.803,0.778,0.759,0.428,0.841,0.700
4,0.833,0.778,0.764,0.586,0.820,0.716
5,0.850,0.722,0.779,0.399,0.789,0.597
Promedio,0.824,0.736,0.752,0.427,0.797,0.634


Calculos globales

In [ ]:
accuracy_global = accuracy_score(
    y_train,
    y_pred_global
)

precision_global, recall_global, _ = precision_recall_curve(
    y_train,
    y_prob_global
)

auprc_global = auc(
    recall_global,
    precision_global
)

aucroc_global = roc_auc_score(
    y_train,
    y_prob_global
)


In [27]:
#Esto es medio feo, pero para meterlo a misma tabla
results_df.loc["Global"] = [
    np.nan,
    accuracy_global,
    np.nan,
    auprc_global,
    np.nan,
    aucroc_global
]

display(results_df.round(4))

,Accuracy train,Accuracy validation,AUPRC train,AUPRC validation,AUCROC train,AUCROC validation
Fold,,,,,,
1,0.8028,0.7000,0.7046,0.3694,0.8034,0.5793
2,0.8306,0.7000,0.7553,0.3525,0.7326,0.5787
3,0.8028,0.7778,0.7593,0.4275,0.8410,0.7003
4,0.8333,0.7778,0.7641,0.5858,0.8197,0.7163
5,0.8500,0.7222,0.7786,0.3986,0.7887,0.5966
Promedio,0.8239,0.7356,0.7524,0.4268,0.7971,0.6343
Global,NaN,0.7356,NaN,0.3977,NaN,0.6390


In [ ]:
#pd (postdata): esto nos deja un indexacion mixta para este df: numeros del 1 al 5 (folds) y string "Promedio" y "Global"!


Faltan los scores finales, hay q ver q vimos en clase no me acuerdo. Los meto despues de leer las teos.

### 2.3 Evaluar configuraciones

In [28]:
from sklearn.model_selection import ParameterGrid

Pongo parametros a usar en una grilla para usar ParameterGrid

In [29]:
parametros = {
    "max_depth": [3, 5, None],
    "criterion": ["gini", "entropy"]
}


In [30]:
pg = list(ParameterGrid(parametros))


Hago algo parecido al punto 2.2

In [31]:
resultados = [] # para guardar los resultados

for param in pg: #uso los parametros guardados en pg

    acc_train_folds = []
    acc_val_folds = []

    for train_idx, val_idx in kf.split(X_train, y_train): #evalua config en los 5 folds

        X_train_fold = X_train.iloc[train_idx]      
        X_val_fold = X_train.iloc[val_idx]         

        y_train_fold = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        #nota gian: medio raro esto
        model = DecisionTreeClassifier(**param) # hago **param asi no tengo que escribir los 6 arboles manualmente

        model.fit(X_train_fold, y_train_fold)

        y_pred_train = model.predict(X_train_fold)
        y_pred_val = model.predict(X_val_fold)

        acc_train_folds.append(                             # guardo accuracy sobre el conj de entrenamiento de esta iteracion
            accuracy_score(y_train_fold, y_pred_train)
        )

        acc_val_folds.append(                                  # guardo accuracy sobre el fold de validacion de esta iteracion
            accuracy_score(y_val_fold, y_pred_val)
        )

    resultados.append({                         # guardo los resultados
        "max_depth": param["max_depth"],
        "criterion": param["criterion"],
        "Accuracy train": np.mean(acc_train_folds),             # calculo el promedio de la accuracy de train 
        "Accuracy validation": np.mean(acc_val_folds)           # calculo el promedio de la accuracy de val
    })

Despues veo los resultados

In [32]:
resultados_df = pd.DataFrame(resultados)

resultados_df

,max_depth,criterion,Accuracy train,Accuracy validation
0,3.0,gini,0.823889,0.728889
1,5.0,gini,0.897222,0.695556
2,NaN,gini,1.000000,0.677778
3,3.0,entropy,0.780556,0.675556
4,5.0,entropy,0.873333,0.702222
5,NaN,entropy,1.000000,0.640000


## 3: Comparación de Algoritmos

### 3.1: Prueba de algoritmos

- Arboles de decisión:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Espacio de búsqueda: 4 hiperparámetros (la consigna pide mínimo 4).
# Tres controlan el crecimiento/tamaño del árbol (capacidad) y uno el criterio de impureza.
espacio_arbol = {
    "max_depth": [3, 5, 10, 20, None],          # None = sin límite de profundidad (fila "Infinito" del 2.3)
    "criterion": ["gini", "entropy"],            # criterio de impureza para elegir cortes
    "max_leaf_nodes": [5, 10, 20, 50, 100, None],# techo a la cantidad de hojas (regiones); None = sin techo
    "min_samples_split": [2, 5, 10, 20, 50],     # mínimo de muestras en un nodo para permitir un corte
}                                                # valores acotados por el tamaño del dataset (~360 train/fold)

busqueda_arbol = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=SEED),  # random_state fija desempates internos del árbol
    param_distributions=espacio_arbol,
    n_iter=50,              # muestrea 50 de las 300 combinaciones posibles (5*2*6*5)
    scoring="roc_auc",      # métrica pedida por la consigna
    cv=kf,                  # el StratifiedKFold de la celda 23: mismos folds para todos los
                            # algoritmos y configs -> comparación justa entre modelos
    random_state=SEED,      # reproducibilidad del muestreo de combinaciones
    n_jobs=-1,              # paraleliza en todos los núcleos
)

busqueda_arbol.fit(X_train, y_train)   # X_train/y_train = SOLO desarrollo; el control no se toca

print("Mejor configuración:", busqueda_arbol.best_params_)
print("Mejor AUCROC (CV):", busqueda_arbol.best_score_)   # ojo: estimación optimista (sesgo de selección)

# Tabla para el informe: top 10 configuraciones con media y desvío entre folds
resultados_arbol = pd.DataFrame(busqueda_arbol.cv_results_)
resultados_arbol = resultados_arbol.sort_values("rank_test_score")
resultados_arbol[["params", "mean_test_score", "std_test_score"]].head(10)

Mejor configuración: {'min_samples_split': 20, 'max_leaf_nodes': 50, 'max_depth': 3, 'criterion': 'gini'}
Mejor AUCROC (CV): 0.6393048076923077


,params,mean_test_score,std_test_score
40,"{'min_samples_split': 10, 'max_leaf_nodes': 20...",0.639305,0.057130
27,"{'min_samples_split': 20, 'max_leaf_nodes': 50...",0.639305,0.057130
5,"{'min_samples_split': 2, 'max_leaf_nodes': 50,...",0.636901,0.058795
15,"{'min_samples_split': 5, 'max_leaf_nodes': 50,...",0.636901,0.058795
3,"{'min_samples_split': 20, 'max_leaf_nodes': 50...",0.627658,0.053334
47,"{'min_samples_split': 20, 'max_leaf_nodes': 10...",0.619237,0.046291
38,"{'min_samples_split': 10, 'max_leaf_nodes': 10...",0.618216,0.052867
37,"{'min_samples_split': 2, 'max_leaf_nodes': 10,...",0.618216,0.052867
7,"{'min_samples_split': 20, 'max_leaf_nodes': 10...",0.618216,0.052867
34,"{'min_samples_split': 5, 'max_leaf_nodes': 10,...",0.618216,0.052867


- KNN:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline: estandarización + KNN como un solo estimador.
# KNN usa distancias, y sin escalar las features de mayor rango dominan el cálculo.
# Al ir dentro del pipeline, el scaler se ajusta SOLO con el train de cada fold
# (media y desvío del train, aplicados a validación) -> sin data leakage.
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])

# Prefijo "knn__" = "este parámetro es del paso llamado knn del pipeline"
espacio_vecinos = {
    "knn__n_neighbors": [3, 10, 50, 150, 250],   # de k chico (flexible, riesgo de overfit) a k grande (suave, riesgo de underfit). tope < ~360 muestras de train por fold
    "knn__weights": ["uniform", "distance"],     # voto parejo vs. ponderado por cercanía
    "knn__p": [1, 2],                            # métrica Minkowski: 1=Manhattan, 2=euclídea; en alta dimensión (200 features) la métrica puede afectar cuán informativas son las distancias
}

busqueda_vecinos = RandomizedSearchCV(
    estimator=pipe_knn,          # el pipeline entero, no el KNN suelto
    param_distributions=espacio_vecinos,
    n_iter=20,                   # el espacio tiene exactamente 5*2*2 = 20 combinaciones: con n_iter=20 la búsqueda es en la práctica exhaustiva (declararlo en el informe)
    scoring="roc_auc",
    cv=kf,                       # mismos folds que el árbol -> comparación justa
    random_state=SEED,
    n_jobs=-1,
)

busqueda_vecinos.fit(X_train, y_train)

print("Mejor configuración:", busqueda_vecinos.best_params_)
print("Mejor AUCROC (CV):", busqueda_vecinos.best_score_)

resultados_vecinos = pd.DataFrame(busqueda_vecinos.cv_results_)
resultados_vecinos = resultados_vecinos.sort_values("rank_test_score")
resultados_vecinos[["params", "mean_test_score", "std_test_score"]].head(10)

Mejor configuración: {'knn__weights': 'distance', 'knn__p': 2, 'knn__n_neighbors': 50}
Mejor AUCROC (CV): 0.778035576923077


,params,mean_test_score,std_test_score
11,"{'knn__weights': 'distance', 'knn__p': 2, 'knn...",0.778036,0.078799
10,"{'knn__weights': 'uniform', 'knn__p': 2, 'knn_...",0.772293,0.084926
9,"{'knn__weights': 'distance', 'knn__p': 1, 'knn...",0.755078,0.070223
8,"{'knn__weights': 'uniform', 'knn__p': 1, 'knn_...",0.745173,0.078898
15,"{'knn__weights': 'distance', 'knn__p': 2, 'knn...",0.723120,0.049102
5,"{'knn__weights': 'distance', 'knn__p': 1, 'knn...",0.721727,0.056510
4,"{'knn__weights': 'uniform', 'knn__p': 1, 'knn_...",0.715041,0.054747
7,"{'knn__weights': 'distance', 'knn__p': 2, 'knn...",0.714537,0.059481
14,"{'knn__weights': 'uniform', 'knn__p': 2, 'knn_...",0.712083,0.047068
6,"{'knn__weights': 'uniform', 'knn__p': 2, 'knn_...",0.705317,0.048448


- SVM:

In [40]:
from sklearn.svm import SVC

# #Claudio dijo que es lo mismo que SVM. SVM también usa distancias/productos internos, así que sufre las escalas igual -> scaler dentro del pipeline (sin leakage).
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(random_state=SEED)),   # kernel RBF por defecto
])

# Los dos hiperparámetros clásicos de SVC con kernel RBF (la consigna pide mínimo 2):
espacio_svm = {
    "svm__C": [0.01, 0.1, 1, 10, 100],           # regularización: C chico = margen blando (más regularizado), C grande = castiga fuerte los errores de train (riesgo de overfit)
    "svm__gamma": ["scale", 0.001, 0.01, 0.1, 1] # alcance del kernel RBF: gamma chico = influencia amplia (suave), gamma grande = influencia local (riesgo de overfit)
}

busqueda_svm = RandomizedSearchCV(
    estimator=pipe_svm,
    param_distributions=espacio_svm,
    n_iter=20,               # 5*5 = 25 combinaciones. Muestrea 20
    scoring="roc_auc",       # usa decision_function por detrás: no hace falta probability=True
    cv=kf,                   # mismos folds que árbol y KNN -> comparación justa
    random_state=SEED,
    n_jobs=-1,
)

busqueda_svm.fit(X_train, y_train)

print("Mejor configuración:", busqueda_svm.best_params_)
print("Mejor AUCROC (CV):", busqueda_svm.best_score_)

resultados_svm = pd.DataFrame(busqueda_svm.cv_results_)
resultados_svm = resultados_svm.sort_values("rank_test_score")
resultados_svm[["params", "mean_test_score", "std_test_score"]].head(10)

Mejor configuración: {'svm__gamma': 0.01, 'svm__C': 100}
Mejor AUCROC (CV): 0.7728221153846154


,params,mean_test_score,std_test_score
16,"{'svm__gamma': 0.01, 'svm__C': 10}",0.772822,0.054865
8,"{'svm__gamma': 0.01, 'svm__C': 100}",0.772822,0.054865
10,"{'svm__gamma': 0.01, 'svm__C': 0.01}",0.769929,0.057682
11,"{'svm__gamma': 0.01, 'svm__C': 1}",0.769580,0.058936
15,"{'svm__gamma': 'scale', 'svm__C': 100}",0.752811,0.040695
12,"{'svm__gamma': 'scale', 'svm__C': 10}",0.752811,0.040695
2,"{'svm__gamma': 'scale', 'svm__C': 0.01}",0.737819,0.053107
9,"{'svm__gamma': 'scale', 'svm__C': 0.1}",0.737699,0.054311
17,"{'svm__gamma': 0.001, 'svm__C': 100}",0.701597,0.029557
1,"{'svm__gamma': 0.001, 'svm__C': 10}",0.689010,0.024581
